In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
import lightgbm as lgb
import xgboost as xgb
from scipy.stats import poisson, spearmanr
from sklearn.calibration import CalibratedClassifierCV
from pandas.api.types import CategoricalDtype
from sklearn.metrics import log_loss, accuracy_score
from typing import List, Tuple, Dict, Generator
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import log_loss, accuracy_score
import warnings
from pathlib import Path
import json
import gc
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)


TRAIN_MODE = True
COMP_TYPE = f"{COMP_TYPE}"
DATA_PATH = f"../../../data/{COMP_TYPE}/"
DIR_PATH_SUBSET = f"../../../outputs/{COMP_TYPE}/feature_selection_subset/"
DIR_PATH_METRIC_RESULT = f"../../../outputs/{COMP_TYPE}/model_metrics/"
DIR_PATH_PICKLE_FILE = f"../../../outputs/{COMP_TYPE}/models/"

gc.collect()
df = pd.read_parquet(DATA_PATH + "club_features_full.parquet")
df.head()

match_id        match_datetime_utc season_id  home_team_id  away_team_id  \
0  12437642 2024-08-23 19:30:00+00:00     61643          2833          2819   
1  12437643 2025-01-24 20:00:00+00:00     61643          6577          2820   
2  12437647 2024-08-25 17:00:00+00:00     61643          2845          6577   
3  12437652 2024-08-24 17:00:00+00:00     61643          2817          2825   
4  12437656 2024-08-24 15:00:00+00:00     61643          2820          2826   

   home_score  away_score  home_venue_id  home_referee_id  \
0         1.0         2.0           86.0         351144.0   
1         1.0         1.0           88.0          86508.0   
2         2.0         1.0          102.0          92296.0   
3         2.0         1.0           80.0          86508.0   
4         1.0         0.0           93.0         137895.0   

   home_macro_rolling_goals  home_macro_rolling_fouls  \
0                      3.36                     25.50   
1                      3.12                     22.70   
2                      2.80                     23.88   
3                      3.10                     24.18   
4                      3.18                     24.88   

   home_macro_rolling_yellows  home_team_avg_goals_last_5  \
0                        4.40                         1.0   
1                        4.22                         0.6   
2                        4.24                         1.0   
3                        4.06                         2.2   
4                        4.10                         1.8   

   home_team_avg_xg_last_5  home_team_avg_buildup_last_5  \
0                    1.446                          87.8   
1                    1.105                         110.8   
2                    0.980                          62.0   
3                    1.910                         135.0   
4                    1.194                         105.4   

   home_team_avg_conceded_last_5  home_team_clean_sheet_pct_last_5  \
0                            2.0                               0.0   
1                            2.0                               0.4   
2                            1.0                               0.4   
3                            0.4                               0.6   
4                            1.2                               0.0   

   home_team_avg_xga_last_5  home_team_avg_ppda_last_5  \
0                     2.034                  10.634066   
1                     2.230                  10.030864   
2                     1.740                  10.492063   
3                     1.070                  10.006186   
4                     1.296                  11.057341   

   home_team_avg_yellows_last_5  home_opponent_avg_yellows_last_5  \
0                           3.2                               1.8   
1                           2.2                               2.4   
2                           3.5                               1.5   
3                           2.2                               1.8   
4                           2.6                               2.0   

   home_team_avg_reds_last_5  home_opponent_avg_reds_last_5  \
0                        0.0                            0.0   
1                        0.2                            0.0   
2                        0.0                            0.0   
3                        0.0                            0.0   
4                        0.0                            0.0   

   home_team_avg_cards_per_foul_last_5  home_team_avg_fouls_committed_last_5  \
0                             0.244048                                  11.6   
1                             0.181392                                  12.2   
2                             0.223982                                  14.0   
3                             0.183020                                  11.0   
4                             0.190476                                  12.4   

   home_team_avg_fouls_drawn_last_5  home_tea

In [27]:
### Eliminar matches que fueron a OT
tms_match_stats = pd.read_parquet(DATA_PATH + 'team_match_stats.parquet')

# Inspect
# tms_match_stats['period'].value_counts()
# Filtering
matches_with_extra_time = (
    tms_match_stats.loc[
        (tms_match_stats["period"] == "ET1") &
        (tms_match_stats["possession_percentage"].notna()),
        "match_id"
    ]
    .unique()
)

DOUBLE_LEG_CROSS_TOURNAMENT = [
    "uefa-europa-league",
    "uefa-champions-league",
    "conference-league",
    "copa_de_italia",
    "copa-del-rey",
    "carabao-cup",
]


df_filtered = df[
    ~(
        df["match_id"].isin(matches_with_extra_time) &
        df["tournament"].isin(DOUBLE_LEG_CROSS_TOURNAMENT)
    )
]
print(f"The Number of matches eliminated on this filter is: {df.shape[0] - df_filtered.shape[0]}")

df_filtered.shape


The Number of matches eliminated on this filter is: 93


(14069, 1410)

,team_match_stat_id,match_id,team_id,is_home_team,period,formation,average_team_rating,total_team_market_value_eur,possession_percentage,big_chances,total_shots,saves,corners,fouls,passes_successful,passes_total,passes_percentage,tackles_successful,tackles_total,tackles_won_percentage,free_kicks,yellow_cards,red_cards,shots_on_target,hit_woodwork,shots_off_target,blocked_shots,shots_inside_box,shots_outside_box,big_chances_missed,fouled_final_third,offsides,accurate_passes_percentage,throw_ins,through_balls,final_third_entries,final_third_passes_successful,final_third_passes_total,final_third_passes_percentage,long_balls_successful,long_balls_total,long_balls_percentage,crosses_successful,crosses_total,crosses_percentage,duels_won_successful,duels_won_total,duels_won_percentage,dispossessed,ground_duels_successful,ground_duels_total,ground_duels_percentage,aerial_duels_successful,aerial_duels_total,aerial_duels_percentage,dribbles_successful,dribbles_total,dribbles_percentage,interceptions,clearances,goal_kicks,expected_goals,touches_in_penalty_area,passes_in_final_third,recoveries,errors_lead_to_shot,goals_prevented,big_saves,errors_lead_to_goal,penalty_saves,big_chances_scored,created_at
0,91253,9541708,2673,True,ALL,4-3-3,6.69,516852000.0,0.49,3,8,2,5,13,364.0,456.0,0.7982,14.0,27.0,0.5185,12,2,0,5,0,3,0,5,3,3,4,4,0.7982,19,2,49,NaN,NaN,NaN,21.0,41.0,0.5122,1.0,14.0,0.0714,53,103,0.51,4,47.0,85.0,0.5529,6.0,18.0,0.3333,8.0,18.0,0.4444,12,10,5,NaN,0,0,0,0,NaN,0,0,0,0,2025-09-02 21:33:45.103854
1,91255,9541708,2673,True,1ST,None,NaN,NaN,0.49,2,4,0,2,0,184.0,229.0,0.8035,3.0,9.0,0.3333,6,2,0,3,0,1,0,3,1,2,1,2,0.8035,12,1,21,NaN,NaN,NaN,13.0,21.0,0.6190,0.0,6.0,0.0000,24,47,0.51,2,19.0,39.0,0.4872,5.0,8.0,0.6250,4.0,7.0,0.5714,4,7,3,NaN,0,0,0,0,NaN,0,0,0,0,2025-09-02 21:33:45.103854
2,91256,9541708,2672,False,1ST,None,NaN,NaN,0.51,4,10,3,4,0,190.0,230.0,0.8261,0.0,5.0,0.0000,10,1,0,2,0,5,3,10,0,3,2,0,0.8261,9,1,23,NaN,NaN,NaN,24.0,39.0,0.6154,6.0,12.0,0.5000,23,47,0.49,4,20.0,39.0,0.5128,3.0,8.0,0.3750,6.0,11.0,0.5455,7,7,1,NaN,0,0,0,0,NaN,0,0,0,1,2025-09-02 21:33:45.103854
3,91257,9541708,2673,True,2ND,None,NaN,NaN,0.49,1,4,2,3,0,180.0,227.0,0.7930,11.0,18.0,0.6111,6,0,0,2,0,2,0,2,2,1,3,2,0.7930,7,1,28,NaN,NaN,NaN,8.0,20.0,0.4000,1.0,8.0,0.1250,29,56,0.52,2,28.0,46.0,0.6087,1.0,10.0,0.1000,4.0,11.0,0.3636,8,3,2,NaN,0,0,0,0,NaN,0,0,0,0,2025-09-02 21:33:45.103854
4,91258,9541708,2672,False,2ND,None,NaN,NaN,0.51,2,7,1,2,0,191.0,242.0,0.7893,6.0,9.0,0.6667,3,2,0,4,0,3,0,3,4,0,1,0,0.7893,11,0,20,NaN,NaN,NaN,15.0,28.0,0.5357,0.0,4.0,0.0000,27,56,0.48,11,18.0,46.0,0.3913,9.0,10.0,0.9000,7.0,14.0,0.5000,4,12,5,NaN,0,0,0,0,NaN,0,0,0,2,2025-09-02 21:33:45.103854
